In [3]:
from geoscholar import db
from geoscholar.ingest.download import pdf_path
from geoscholar.ingest.parse import sections_from_tei, grobid_tei, sections_from_pymupdf

In [5]:
# GROBID vs PyMuPDF

from geoscholar import db
from geoscholar.ingest.download import pdf_path
from geoscholar.ingest.parse import sections_from_tei, grobid_tei, sections_from_pymupdf

conn = db.connect()
for p in db.papers_with_status(conn, "downloaded"):
    path = pdf_path(p["id"])
    g = sections_from_tei(grobid_tei(path, p["id"]))
    m = sections_from_pymupdf(path)
    print(f"\n{p['id']}  {p['title'][:60]}")
    print(f"  GROBID : {len(g)} sections, {sum(len(s['text']) for s in g):,} chars")
    print("           ", [s["heading"][:25] for s in g][:8])
    print(f"  PyMuPDF: {sum(len(s['text']) for s in m):,} chars, no headings")


2610.23377  FloodVLM-Pix: Language-Guided Pixel-Level Flood Understandin
  GROBID : 14 sections, 27,727 chars
            ['Abstract', '1. Introduction', '2.1. Flood Model/Benchmar', '2.2. Remote Sensing VLM/B', '3. Benchmark', '4.1. Overview of FloodVLM', '4.2. Prior Module and VLM', '4.3. Pixel Decoder']
  PyMuPDF: 33,583 chars, no headings

2610.23416  GeoReason: Bridging Logical Reasoning and Spatial Fidelity i
  GROBID : 27 sections, 30,829 chars
            ['Abstract', '1 Introduction', 'Anticipatory Prior via Pr', 'Enhanced Visual Reasoning', '2 Related Work', 'Visual Reasoning in MLLMs', '3 Our Method', '3.1 System Architecture']
  PyMuPDF: 29,715 chars, no headings

2610.23440  ShadowCast: On-the-Fly Context-Aware Shadow Augmentation for
  GROBID : 21 sections, 41,402 chars
            ['Abstract', 'R EMOTE sensing (RS) imag', 'III. PROPOSED METHOD A. M', 'B. Method', 'A. Experimental Setup', 'Augmentation Vaihingen Po', 'B. Ablation Study 1) Para', 'C. Comparison across Seg

In [17]:
# Cell 2: inspect one paper's sections
sections = sections_from_tei(grobid_tei(path, p["id"]))
for s in sections:
    print(p["id"])
    print(f"--- {s['heading']} ({len(s['text'])} chars)")
    if s['heading'] == '3. Methodology': 
        print(s["text"][:], "...\n") 
    else:
        print(s["text"][:300], "...\n")

2610.24075
--- Abstract (546 chars)
DTM RMSE a : 2.35 m DTM RMSE a : 2.82 m DTM RMSE a : 16.18 m DTM RMSE a : 0.43 m a OrTeN DTM (m) b Training image c OrTeN render d Shadows disabled Figure 1. Summary of OrTeN DTM reconstructions. OrTeN produces high-quality digital terrain maps (DTMs) from multi-view imagery without structural prior ...

2610.24075
--- 1. Introduction (3555 chars)
Digital terrain maps (DTMs) are an important supporting data product in space exploration. Accurate DTMs enable landing site selection and terrain-relative navigation (TRN) during entry, descent, and landing for modern autonomous space exploration missions such as Mars 2020 [11,21]. They also provid ...

2610.24075
--- 2. Related Work DTM creation with MVS. (3497 chars)
A number of well-established pipelines exist for creating DTMs from orbital imagery using MVS, including the Ames Stereo Pipeline (ASP), satellite stereo pipeline (s2p), and COLMAP [7,13,33,34]. All of these methods use feature extraction an

In [19]:
# Cell 3: chunk one paper with the real tokenizer
from sentence_transformers import SentenceTransformer
from geoscholar import config
from geoscholar.ingest.chunk import chunk_paper

tok = SentenceTransformer(config.EMBED_MODEL).tokenizer
count = lambda t: len(tok.encode(t, add_special_tokens=False))

for size in (150, 300, 500):
    chunks = chunk_paper(p, sections, max_tokens=size, overlap=50, count=count)
    sizes = [c["n_tokens"] for c in chunks]
    print(f"max {size}: {len(chunks)} chunks, avg {sum(sizes) / len(sizes):.0f} tokens, max {max(sizes)}")

chunks = chunk_paper(p, sections, count=count)      # default 300 / 50
print("\n" + chunks[3]["embed_text"][:600])

/home/jthe/Dev/tutes/geoscholar/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6652.32it/s]


max 150: 49 chunks, avg 114 tokens, max 150
max 300: 27 chunks, avg 188 tokens, max 290
max 500: 18 chunks, avg 281 tokens, max 469

OrTeN: Neural Inverse Graphics for Joint Terrain and Line-scan Camera Trajectory Reconstruction
Section: 1. Introduction

OrTeN mitigates the shortcomings of classical MVS and volumetric neural reconstruction by optimizing a textured height map directly. Critically, OrTeN requires no structural priors to achieve accurate terrain reconstruction from images and approximate camera poses. It builds on Neural Terrain Maps (NTM) [8], adding direct bundle adjustment of line-scan camera extrinsics jointly with training, a semi-empirical photometric model, and geometric shadow modeling. The contributio


In [23]:
# Cell 4: index the downloaded papers and search them
from geoscholar.index import Embedder, Index
from geoscholar.ingest.parse import parse_pdf

embedder = Embedder()                 # loads the model once; reuse it
index = Index(embedder)
index.ensure_collection()

for p in db.papers_with_status(conn, "downloaded"):
    sections, parser = parse_pdf(pdf_path(p["id"]), p["id"])
    chunks = chunk_paper(p, sections, count=embedder.count_tokens)
    n = index.replace_paper(p["id"], chunks)
    print(p["id"], parser, n, "chunks")

print("total points:", index.count())

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 17846.19it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (541 > 512). Running this sequence through the model will result in indexing errors


2610.23377 grobid 31 chunks
2610.23416 grobid 37 chunks
2610.23440 grobid 57 chunks
2610.23609 grobid 24 chunks
2610.24075 grobid 27 chunks
total points: 176


In [25]:
# Cell 5: search
for score, hit in index.search("what dataset was used for evaluation?", k=5):
    print(f"{score:.3f}  {hit['paper_id']}  [{hit['section']}]  {hit['text'][:100]}")

0.635  2610.23609  [I. INTRODUCTION]  In contrast to general computer vision (CV) tasks, which are supported by widely adopted datasets su
0.621  2610.23609  [V. CONCLUSION]  The ADA 2025 dataset fills a gap in existing data supply for cutting-edge crop management, providing
0.614  2610.24075  [4.1. Datasets]  1.51 / 1.46 [5] Table 1. Dataset instrument and DTM reference overview. Instruments are a CTX [25] a
0.611  2610.23609  [III. EXEMPLARY APPLICATIONS AND RESULTS]  The code for the exemplary analyses below is available on GitHub [20]. All models and correlations u
0.610  2610.23609  [IV. LIMITATIONS]  As is the case with any large-scale dataset, there are also time points in the dataset where a senso


In [27]:
# Cell 6: connect to the full index
import textwrap, time, requests
from geoscholar.index import Embedder, Index

embedder = Embedder()
index = Index(embedder)
print("chunks in collection:", index.count())

def show(hits):
    for score, h in hits:
        print(f"{score:.3f}  {h['paper_id']} ({h['year']})  [{h['section'][:30]}]")
        print("       ", h["text"][:140].replace("\n", " "), "...")

show(index.search("which loss function is used for training the segmentation network?", k=5))

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 15621.97it/s]


chunks in collection: 19650
0.800  2608.26489 (2026)  [B. Loss function]
        The equal optimization of both precision and recall aims to maximize the F 1 score, a common metric to rank the performance of segmentation  ...
0.776  2608.26489 (2026)  [I. INTRODUCTION]
        Deep learning approaches circumvent this singular pixel context window limitation by pooling information spatially at different spatial reso ...
0.776  2608.15054 (2026)  [D. Loss Function]
        We employ a composite loss function which combines the Binary Cross Entropy (BCE) loss, the Dice loss, and the L1 loss as follows: where S g ...
0.762  2608.15683 (2026)  [Segmentation Loss]
        Loss functions play a crucial role in the training of segmentation models. Among them, cross-entropy loss is a widely used pixel-wise measur ...
0.760  2608.26489 (2026)  [APPENDIX C SCALED LOSS COMPARI]
        Observing Fig. 12 it can be seen that the proposed loss function attains marginally greater recall scores whilst 

In [29]:
# Cell 7: metadata filter, only papers from this year onward
show(index.search("SAR image despeckling", k=5, year_from=2026))

0.838  2608.11271 (2026)  [B. SAR Despeckling]
        All these methods consider compressing and reconstructing SAR images as a full signal, amplitude and phase. However, allowing the loss of th ...
0.827  2608.15028 (2026)  [2.1. Learning-Based Image Rest]
        Supervised image restoration learns a mapping from degraded images to clean references. Residual CNNs provide established baselines for natu ...
0.822  2608.15028 (2026)  [1. Introduction]
        Image restoration aims to suppress acquisition noise while preserving edges, textures, and weak structures. This task is particularly challe ...
0.810  2608.11142 (2026)  [2.2 Pretext Tasks for SAR Inte]
        Early SAR SSL targeted despeckling because of the presence of multiplicative speckle noise. Boulch et al. (2018) showed that cross-acquisiti ...
0.804  2608.15028 (2026)  [4.3. Results on Real SAR Image]
        The real-data study covers urban and natural scenes from six SAR configurations. Representative results for medi

In [37]:
# Cell 7b: LLM settings and chat() helper (copied from week 1)
OLLAMA_URL = "http://localhost:11434"
LLM_MODEL = "llama3.1:8b"          # use the exact name shown by `ollama list`

def chat(prompt, system=None, temperature=0.1):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": LLM_MODEL, "messages": messages, "stream": False,
              "options": {"temperature": temperature}},
        timeout=300,
    )
    if not resp.ok:
        raise RuntimeError(f"Ollama error {resp.status_code}: {resp.text}")
    return resp.json()["message"]["content"]

In [39]:
# Cell 8: RAG over full text with section citations
SYSTEM_PROMPT = """You are a research assistant for remote sensing literature.
Answer using ONLY the context passages below.
Cite every claim as [paper_id, section], e.g. [2410.01234, 3 Method].
If the context does not contain the answer, reply exactly:
"I could not find this in the indexed papers."
Be concise."""

def build_prompt(question, hits):
    blocks = [f"[{h['paper_id']}, {h['section']}] {h['title']} ({h['year']})\n{h['text']}"
              for _, h in hits]
    return "Context:\n" + "\n\n---\n\n".join(blocks) + f"\n\nQuestion: {question}\nAnswer:"

def ask(question, k=5, year_from=None):
    t0 = time.time()
    hits = index.search(question, k=k, year_from=year_from)
    t1 = time.time()
    answer = chat(build_prompt(question, hits), system=SYSTEM_PROMPT)
    t2 = time.time()
    print(textwrap.fill(answer, 100))
    print("\nSources:")
    for score, h in hits:
        print(f"  {score:.3f}  [{h['paper_id']}, {h['section'][:30]}]  https://arxiv.org/abs/{h['paper_id']}")
    print(f"\nretrieval {t1 - t0:.2f} s | generation {t2 - t1:.1f} s")
    return answer, hits

answer, hits = ask("What evaluation metrics do papers use for change detection, and on which datasets?")

Based on the provided context, the evaluation metrics used for change detection are as follows:  For
Binary Change Detection (BCD):  * F1-score [2608.24263, Datasets and Evaluation Metrics:] * IoU
[2608.24263, Datasets and Evaluation Metrics:] * BCDS [2607.27969, 3) Evaluation metrics:]  For
Semantic Change Detection (SCD):  * F1-score [2608.24263, Datasets and Evaluation Metrics:] * mIoU
[2608.24263, Datasets and Evaluation Metrics:] * SCS [2608.24263, Datasets and Evaluation Metrics:]
* SeK [2608.24263, Datasets and Evaluation Metrics:]  The datasets used for evaluation are:  *
LEVIR-CD [18] * WHU-CD [19] * DSIFN-CD [20] * SEC-BCD [16] * SECOND * HRSCD * FootprintNet uses the
following datasets:         + LEVIR-CD [18]         + WHU-CD [19]         + SEC-BCD [16]         +
SECOND         + HRSCD

Sources:
  0.744  [2608.28247, 2.2. Trustworthy and reproduci]  https://arxiv.org/abs/2608.28247
  0.740  [2608.24263, Datasets and Evaluation Metric]  https://arxiv.org/abs/2608.24263
  0.7